In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
from datetime import datetime, timezone
from pyspark.sql import functions as F


catalog_name = spark.sql("SELECT current_catalog()").first()[0]
schema_name = spark.sql("SELECT current_schema()").first()[0]

RUN_REGISTRY_TABLE = (
    f"{catalog_name}.{schema_name}.fraud_model_run_registry"
)

latest_run = (
    spark.table(RUN_REGISTRY_TABLE)
    .orderBy(F.desc("registered_at_utc"))
    .first()
)

RUN_ID = latest_run["run_id"]

print("Using MLflow run:", RUN_ID)

TEST_TABLE = f"{catalog_name}.{schema_name}.fraud_transaction_test"
PREDICTIONS_TABLE = f"{catalog_name}.{schema_name}.fraud_predictions_test"

test_pdf = spark.table(TEST_TABLE).toPandas()

X_test = test_pdf.drop(["Class"], axis=1)
y_test = test_pdf["Class"]

print("Test rows:", len(X_test))
print("Fraud cases:", int(y_test.sum()))

{"ts": "2026-10-11 10:08:04.429", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzIxNTM3NjI5NDUwNjIzMxABIAEyJDAxYTEyYTZlLWZmMWYtNzQ1OC1hMjAyLTkwNThiYWE3YTQzYTokNjMwYzRhZGQtNGJjZi0zZTE5LTkwNTItZTljMGM2Mzk5NThlSgwI/7+t1gYQwJ/G8QJQAVgBYAFo0t++xovFow2IAQA=.", "context": {}}
{"ts": "2026-10-11 10:08:04.429", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzIxNTM3NjI5NDUwNjIzMxABIAEyJDAxYTEyYTZlLWZmMWYtNzQ1OC1hMjAyLTkwNThiYWE3YTQzYTokNjMwYzRhZGQtNGJjZi0zZTE5LTkwNTItZTljMGM2Mzk5NThlSgwI/7+t1gYQwJ/G8QJQAVgBYAFo0t++xovFow2IAQA=.", "context": {}}
{"ts": "2026-10-11 10:08:04.429", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzIxNTM3NjI5NDUwNjIzMxABIAEyJDAxYTEyYTZlLWZmMWYtNzQ1OC1hMjAyLTkwNThiYWE3YTQzYTokNjMwYzRhZGQtNGJjZi0zZTE5LTkwNTItZTljMGM2

Using MLflow run: b3ca314766bc4afbb27d9bb58180dfbf
Test rows: 56962
Fraud cases: 98


In [0]:
MODEL_URI = f"runs:/{RUN_ID}/fraud_model"

model = mlflow.sklearn.load_model(MODEL_URI)

print("Model Loaded")
print("Model type:", type(model).__name__)

Model Loaded
Model type: Pipeline


In [0]:
THRESHOLD = 0.5

X_test = X_test[model.feature_names_in_] if hasattr(
    model, "feature_names_in_"
) else X_test

fraud_probablity = model.predict_proba(X_test)[:, 1]
fraud_prediction = (fraud_probablity >= THRESHOLD).astype(int)

predictions_pdf = X_test.copy()
predictions_pdf["actual_class"] = y_test.to_numpy()
predictions_pdf["fraud_probability"] = fraud_probablity
predictions_pdf["predicted_class"] = fraud_prediction
predictions_pdf["threshold"] = THRESHOLD
predictions_pdf["model_run_id"] = RUN_ID
predictions_pdf["scored_at_utc"] = datetime.now(timezone.utc).isoformat()

print("Predictions generated:", len(predictions_pdf))
print("Transactions flagged:", int(fraud_prediction.sum()))
print("Actual fraud cases:", int(y_test.sum()))
      


Predictions generated: 56962
Transactions flagged: 1476
Actual fraud cases: 98


In [0]:
from sklearn.metrics import(
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    average_precision_score
)

batch_metrics = {
    "precision": precision_score(y_test, fraud_prediction, zero_division=0),
    "recall": recall_score(y_test, fraud_prediction, zero_division=0),
    "f1": f1_score(y_test, fraud_prediction, zero_division=0),
    "roc_auc": roc_auc_score(y_test, fraud_prediction),
    "average_precision": average_precision_score(y_test, fraud_probablity),
}

for name, value in batch_metrics.items():
    print(f"{name}: {value:.4f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_test, fraud_prediction))


precision: 0.0610
recall: 0.9184
f1: 0.1144
roc_auc: 0.9470
average_precision: 0.7159

Confusion matrix:
[[55478  1386]
 [    8    90]]


In [0]:
predictions_sdf = spark.createDataFrame(predictions_pdf)
(predictions_sdf.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(PREDICTIONS_TABLE))

print("Saved predictions to:", PREDICTIONS_TABLE)

Saved predictions to: workspace.default.fraud_predictions_test


In [0]:
saved_predictions = spark.table(PREDICTIONS_TABLE)
print("Saved rows:", saved_predictions.count())

print("Prediction distribution")
display(
    saved_predictions.groupBy("predicted_class")
    .count()
    .orderBy("predicted_class")
)

print("Highest-risk transactions:")
display(
    saved_predictions
    .orderBy(F.desc("fraud_probability"))
    .select(
        "Time",
        "Amount",
        "actual_class",
        "predicted_class",
        "fraud_probability",
        "model_run_id",
    )
    .limit(10)
)

assert saved_predictions.count() == len(test_pdf)

Saved rows: 56962
Prediction distribution


predicted_class,count
0,55486
1,1476


Highest-risk transactions:


Time,Amount,actual_class,predicted_class,fraud_probability,model_run_id
93853.0,316.06,1,1,1.0,b3ca314766bc4afbb27d9bb58180dfbf
84204.0,512.25,1,1,1.0,b3ca314766bc4afbb27d9bb58180dfbf
100298.0,1.0,1,1,1.0,b3ca314766bc4afbb27d9bb58180dfbf
19762.0,1.0,1,1,1.0,b3ca314766bc4afbb27d9bb58180dfbf
146022.0,1.18,1,1,1.0,b3ca314766bc4afbb27d9bb58180dfbf
95628.0,1.63,1,1,1.0,b3ca314766bc4afbb27d9bb58180dfbf
25198.0,99.99,1,1,1.0,b3ca314766bc4afbb27d9bb58180dfbf
35771.0,60.6,1,1,1.0,b3ca314766bc4afbb27d9bb58180dfbf
25231.0,99.99,1,1,1.0,b3ca314766bc4afbb27d9bb58180dfbf
24735.0,89.99,0,1,1.0,b3ca314766bc4afbb27d9bb58180dfbf
